# Spindle → Microsoft Fabric Lakehouse

Generate synthetic data and write it directly to a Fabric Lakehouse as managed Delta tables.

**Prerequisites**
- Running inside a Microsoft Fabric notebook
- A Lakehouse attached to the notebook (`spark` and `notebookutils` are pre-available)

**What this notebook does**
1. Generates the Retail domain at `fabric_demo` scale
2. Writes all 9 tables to the Lakehouse as Delta tables
3. Queries the tables using Spark SQL
4. Writes a star schema (dim + fact tables) to the Lakehouse
5. Optionally writes all 12 domains

In [ ]:
%pip install sqllocks-spindle --quiet

## Generate retail data

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)
print(result)

## Write all tables to Lakehouse as Delta

Each pandas DataFrame is converted to a Spark DataFrame and saved as a managed Delta table in the attached Lakehouse.
Tables appear immediately in the **Lakehouse explorer** and **SQL analytics endpoint**.

In [ ]:
TABLE_PREFIX = "spindle_retail_"  # change or set to "" for no prefix

for table_name, df in result.tables.items():
    full_name = f"{TABLE_PREFIX}{table_name}"
    (
        spark.createDataFrame(df)
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_name)
    )
    print(f"  {full_name:<40} {len(df):>7,} rows")

print(f"\nDone — {len(result.tables)} tables written to Lakehouse.")

## Query with Spark SQL

In [ ]:
spark.sql("SELECT * FROM spindle_retail_customer LIMIT 5").show()

In [ ]:
spark.sql("""
    SELECT
        c.loyalty_tier,
        COUNT(DISTINCT o.order_id)     AS order_count,
        ROUND(SUM(o.order_total), 2)   AS total_revenue,
        ROUND(AVG(o.order_total), 2)   AS avg_order_value
    FROM spindle_retail_order o
    JOIN spindle_retail_customer c ON o.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
    ORDER BY total_revenue DESC
""").show()

## Write star schema to Lakehouse

Transform the 3NF tables into a proper star schema (dim + fact tables) and write each one as a Delta table.
This is ready for Power BI DirectLake mode.

In [ ]:
from sqllocks_spindle.transform import StarSchemaTransform

transform = StarSchemaTransform()
star = transform.transform(result.tables, RetailDomain().star_schema_map())
print(star.summary())

In [ ]:
STAR_PREFIX = "spindle_star_"

for table_name, df in star.all_tables().items():
    full_name = f"{STAR_PREFIX}{table_name}"
    (
        spark.createDataFrame(df)
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_name)
    )
    print(f"  {full_name:<40} {len(df):>7,} rows")

print(f"\nStar schema written — {len(star.dimensions)} dims, {len(star.facts)} facts, date dim included.")

In [ ]:
# Verify SK referential integrity between fact_sale and dim_customer
spark.sql("""
    SELECT
        d.loyalty_tier,
        COUNT(*) AS sales,
        ROUND(SUM(f.unit_price * f.quantity), 2) AS revenue
    FROM spindle_star_fact_sale f
    JOIN spindle_star_dim_customer d ON f.sk_customer = d.sk_customer
    GROUP BY d.loyalty_tier
    ORDER BY revenue DESC
""").show()

## All 12 domains to Lakehouse (optional)

Writes every domain to its own set of Delta tables. At `fabric_demo` scale this takes ~30–60 seconds.

In [ ]:
from sqllocks_spindle import (
    RetailDomain, HealthcareDomain, FinancialDomain, SupplyChainDomain,
    IoTDomain, HRDomain, InsuranceDomain, MarketingDomain,
    EducationDomain, RealEstateDomain, ManufacturingDomain, TelecomDomain,
)

ALL_DOMAINS = [
    ("retail",        RetailDomain()),
    ("healthcare",    HealthcareDomain()),
    ("financial",     FinancialDomain()),
    ("supply_chain",  SupplyChainDomain()),
    ("iot",           IoTDomain()),
    ("hr",            HRDomain()),
    ("insurance",     InsuranceDomain()),
    ("marketing",     MarketingDomain()),
    ("education",     EducationDomain()),
    ("real_estate",   RealEstateDomain()),
    ("manufacturing", ManufacturingDomain()),
    ("telecom",       TelecomDomain()),
]

for domain_name, domain in ALL_DOMAINS:
    r = spindle.generate(domain=domain, scale="fabric_demo", seed=42)
    for table_name, df in r.tables.items():
        full_name = f"spindle_{domain_name}_{table_name}"
        spark.createDataFrame(df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    total = sum(len(r[t]) for t in r.tables)
    print(f"{domain_name:<16} {len(r.tables):>2} tables  {total:>7,} rows")

print("\nAll 12 domains written to Lakehouse.")